In [0]:
MERGE INTO airline_catalog.gold.dim_date AS target
USING (
    SELECT DISTINCT
        flight_date AS date_id,
        YEAR(flight_date) AS year,
        MONTH(flight_date) AS month,
        DAY(flight_date) AS day,
        DAYOFWEEK(flight_date) AS day_of_week,
        CASE 
            WHEN DAYOFWEEK(flight_date) IN (1, 7) THEN true
            ELSE false
        END AS is_weekend,
        CURRENT_TIMESTAMP() AS _created_at
    FROM airline_catalog.silver.flights_silver
    WHERE flight_date IS NOT NULL
) AS source
ON target.date_id = source.date_id
WHEN MATCHED THEN
  UPDATE SET
    target.year = source.year,
    target.month = source.month,
    target.day = source.day,
    target.day_of_week = source.day_of_week,
    target.is_weekend = source.is_weekend,
    target._created_at = source._created_at
WHEN NOT MATCHED THEN
  INSERT (
    date_id, year, month, day, day_of_week, is_weekend, _created_at
  )
  VALUES (
    source.date_id, source.year, source.month, source.day, source.day_of_week, source.is_weekend, source._created_at
  );

In [0]:
SELECT year, month, COUNT(*) AS cantidad_fechas
FROM airline_catalog.gold.dim_date
GROUP BY year, month
ORDER BY year ASC, month ASC

In [0]:
SELECT YEAR(flight_date) AS anio, MONTH(flight_date) AS mes
FROM airline_catalog.silver.flights_silver
GROUP BY anio, mes
ORDER BY anio, mes